# 11 — Função valor V_t(W)

Desenvolve `recorrencia_B` e `funcao_valor`, e confere a Etapa 7 nas trajetórias da esteira. **F11.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
import pandas as pd
from app.principal import executar_pipeline

## Desenvolvimento

As funções abaixo foram escritas e testadas aqui, e depois passaram para app/nucleo.py.

In [2]:
def recorrencia_B(A, phi, beta):
    """B_T=0; B_t = -A_t*ln(A_t) + beta*A_{t+1}*ln(beta*A_{t+1}) + beta*A_{t+1}*Phi + beta*B_{t+1}. (F11)

    So existe no caso gamma=1, em que V_t(W) = A_t*ln(W) + B_t. O B_t junta o
    que nao depende de W e sai de colocar na equacao de Bellman o consumo
    otimo W_t/A_t e a poupanca beta*A_{t+1}*W_t/A_t. O Phi aqui e o do caso
    log, Phi_chapeu = E[ln R_p]. Sao T+1 valores, como os do A.
    """
    A = np.asarray(A, dtype=float)
    T = A.shape[0] - 1
    B = np.zeros(T + 1)
    for t in range(T - 1, -1, -1):
        B[t] = (-A[t] * np.log(A[t]) + beta * A[t + 1] * np.log(beta * A[t + 1])
                + beta * A[t + 1] * phi + beta * B[t + 1])
    return B

In [3]:
def funcao_valor(A, W, gamma, B=None):
    """V_t(W) = A_t*W^(1-gamma)/(1-gamma) (gamma diferente de 1) ou A_t*ln(W) + B_t (gamma=1). (F11)

    Com gamma=1 a funcao valor tem o termo B_t da recorrencia_B, e sem ele o
    resultado so valeria em t=T; por isso o B e obrigatorio nesse caso. Com
    gamma diferente de 1 esse termo nao existe e o B e ignorado.
    """
    A = np.asarray(A, dtype=float)
    if np.isclose(gamma, 1.0):
        if B is None:
            raise ValueError(
                "com gamma=1 a funcao valor precisa do termo B_t "
                "(nucleo.recorrencia_B); sem ele o valor so vale em t=T."
            )
        return A * np.log(W) + np.asarray(B, dtype=float)
    return A * W ** (1.0 - gamma) / (1.0 - gamma)

**Teste**: no último período o A vale 1, então a função valor tem que ser igual à utilidade.

In [4]:
g = 4.0
W = 5.0
V_T = funcao_valor(np.array([1.0]), W, g)[0]
print('V_T(W):', V_T, '| u(W):', W**(1-g)/(1-g))

V_T(W): -0.0026666666666666666 | u(W): -0.0026666666666666666


In [5]:
assert np.isclose(V_T, W**(1-g)/(1-g))

**Teste**: com gamma = 1 e um período só, a conta sai à mão. O consumo ótimo é W/(1+beta), e V_0(W) = ln(W/(1+beta)) + beta*ln(beta*W/(1+beta)) + beta*E[ln R_p]. O A_0 = 1 + beta e o B_0 da recorrência têm que reproduzir esse valor.

In [6]:
beta_1 = 0.96
phi_log = 0.004                      # E[ln R_p], um valor qualquer
A_1p = np.array([1.0 + beta_1, 1.0])
B_1p = recorrencia_B(A_1p, phi_log, beta_1)
W = 2.0
V0_conta = np.log(W/(1+beta_1)) + beta_1*np.log(beta_1*W/(1+beta_1)) + beta_1*phi_log
V0_func = funcao_valor(A_1p, W, 1.0, B=B_1p)[0]

print('V_0 pela conta:', V0_conta, '| pela funcao:', V0_func, '| B:', B_1p)

V_0 pela conta: 0.0042481916028931956 | pela funcao: 0.004248191602893359 | B: [-1.35432028  0.        ]


In [7]:
assert np.isclose(V0_func, V0_conta) and B_1p[-1] == 0.0

In [8]:
try:
    funcao_valor(A_1p, W, 1.0)
    recusou = False
except ValueError:
    recusou = True

assert recusou, 'com gamma=1 e sem o B a funcao tem que recusar'

**Teste** (Etapa 7): a utilidade que a política simulada entrega tem que bater com o V_0(W_0) da função valor. Pela equação de Bellman, para todo t,

V_0(W_0) = sum_{s<t} beta^s E[u(c_s)] + beta^t E[V_t(W_t)],

e em t = T o lado direito é a utilidade esperada da política inteira. As esperanças são as médias que a esteira devolve (`trajetoria_u_media` e `trajetoria_V_media`). O teste roda com gamma = 5 e com gamma = 1, que precisa do B_t. A tolerância é de 2% porque essas médias são de Monte Carlo, e com gamma = 5 o V_t vai com W_t^(-4), o que faz a média demorar para convergir.

In [9]:
rng = np.random.default_rng(7)
ruido = rng.normal(0,0.05,300)
ruido -= ruido.mean()
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=300,freq='MS').strftime('%Y-%m'),
                    'ibov':0.012+ruido,'cdi':np.full(300,0.008)})

def desvio_bellman(res):
    """Maior desvio relativo, em t=0..T, entre os dois lados da identidade acima."""
    b, T = res['beta'], res['horizonte']
    u_med, V_med = res['trajetoria_u_media'], res['trajetoria_V_media']
    desc = b ** np.arange(T + 1)
    acumulado = np.concatenate([[0.0], np.cumsum(desc[:-1] * u_med[:-1])])
    return np.max(np.abs((acumulado + desc * V_med) / V_med[0] - 1))

desvios = {}
for g in (5.0, 1.0):
    res = executar_pipeline({'retornos':ret,'ativos':['ibov'],'periodos_por_ano':12,'cdi_anual':0.10,
                             'gamma':g,'beta_anual':0.96,'w0':1.0,'horizonte':60,
                             'n_paths':20_000,'seed':1})
    desvios[g] = desvio_bellman(res)
    print(f"gamma={g:g}: V_0(W_0)={res['trajetoria_V_media'][0]:.6f} | desvio maximo={desvios[g]:.2e}")

gamma=5: V_0(W_0)=-70525849.640535 | desvio maximo=8.55e-03
gamma=1: V_0(W_0)=-208.337915 | desvio maximo=2.80e-03


In [10]:
assert all(d < 0.02 for d in desvios.values())